In [0]:
dbutils.fs.mkdirs("/FileStore/tables/hospital_data/")

# 1. Dataset: doctors.csv
doctors_data = """doctor_id,doctor_name,specialization,city,experience_years,consultation_fee
D101,Dr. Ramesh,Cardiology,Hyderabad,12,1500
D102,Dr. Priya,Neurology,Bangalore,10,2000
D103,Dr. Anita,Dermatology,Chennai,8,1000
D104,Dr. Suresh,Orthopedics,Mumbai,15,2500
D105,Dr. Meera,Pediatrics,Delhi,6,1200
D106,Dr. Kiran,Cardiology,Hyderabad,20,3000
D107,Dr. Farhan,Neurology,Pune,5,1800
D108,Dr. Nisha,Dermatology,Kochi,9,1100"""

dbutils.fs.put("/FileStore/tables/hospital_data/doctors.csv", doctors_data, overwrite=True)

# 2. Dataset: visits.csv
visits_data = """visit_id,patient_name,doctor_id,visit_date,diagnosis,bill_amount,payment_status
V1001,Rahul Sharma,D101,2026-01-10,Heart Checkup,5000,Paid
V1002,Priya Reddy,D102,2026-01-10,Migraine,3500,Paid
V1003,Amit Kumar,D103,2026-01-11,Skin Allergy,2000,Pending
V1004,Sneha Patel,D104,2026-01-11,Fracture,12000,Paid
V1005,Farhan Ali,D105,2026-01-12,Fever,1500,Paid
V1006,Neha Singh,D106,2026-01-12,Heart Checkup,7000,Paid
V1007,Arjun Verma,D120,2026-01-13,Migraine,4000,Paid
V1008,Meera Nair,D103,2026-01-13,Skin Allergy,,Pending
V1009,Kiran Rao,D104,2026-01-14,Back Pain,6000,Paid
V1010,Nisha Reddy,D101,2026-01-14,Heart Checkup,5500,Paid"""

dbutils.fs.put("/FileStore/tables/hospital_data/visits.csv", visits_data, overwrite=True)

# 3. Dataset: hospital_config.json
import json
json_data = [
    {
        "hospital_id": "H101",
        "hospital_name": "Apollo Hospital",
        "city": "Hyderabad",
        "contact": {"phone": "9876500011", "email": "apollo@mail.com"},
        "services": ["Cardiology", "Neurology", "Dermatology"]
    },
    {
        "hospital_id": "H102",
        "hospital_name": "Yashoda Hospital",
        "city": "Hyderabad",
        "contact": {"phone": None, "email": "yashoda@mail.com"},
        "services": ["Cardiology", "Orthopedics"]
    },
    {
        "hospital_id": "H103",
        "hospital_name": "Care Hospital",
        "city": "Bangalore",
        "contact": {"phone": "9876500013", "email": None},
        "services": ["Neurology", "Pediatrics"]
    }
]

dbutils.fs.put("/FileStore/tables/hospital_data/hospital_config.json", json.dumps(json_data), overwrite=True)
print("Environment set up successfully! Data written to DBFS.")

Wrote 409 bytes.
Wrote 628 bytes.
Wrote 557 bytes.
Environment set up successfully! Data written to DBFS.


CSV Exercises

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import IntegerType, DoubleType

doctors_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/FileStore/tables/hospital_data/doctors.csv")
doctors_df.show()
visits_df = spark.read.option("header", "true").option("inferSchema", "true").csv("/FileStore/tables/hospital_data/visits.csv")
visits_df.show()
print("Doctors Schema:")
doctors_df.printSchema()
print("Visits Schema:")
visits_df.printSchema()
print(f"Total Doctors: {doctors_df.count()}")
print(f"Total Visits: {visits_df.count()}")

display(doctors_df.filter(col("city") == "Hyderabad"))
display(doctors_df.filter(col("specialization") == "Cardiology"))
display(doctors_df.filter(col("experience_years") > 10))
display(visits_df.filter(col("bill_amount") > 5000))
display(visits_df.filter(col("payment_status") == "Pending"))

display(visits_df.filter(col("payment_status") == "Paid"))
display(doctors_df.groupBy("specialization").agg(avg("consultation_fee").alias("avg_consultation_fee")))
display(doctors_df.groupBy("specialization").agg(max("consultation_fee").alias("max_consultation_fee")))
display(doctors_df.groupBy("city").agg(count("doctor_id").alias("doctor_count")))
display(doctors_df.groupBy("specialization").agg(count("doctor_id").alias("doctor_count")))

display(visits_df.agg(sum("bill_amount").alias("total_bill_amount")))
display(visits_df.agg(avg("bill_amount").alias("avg_bill_amount")))
display(visits_df.agg(max("bill_amount").alias("highest_bill_amount")))
display(visits_df.agg(min("bill_amount").alias("lowest_bill_amount")))
display(doctors_df.orderBy(desc("consultation_fee")))

display(visits_df.orderBy(desc("bill_amount")))
display(visits_df.filter(col("bill_amount").isNull()))
visits_filled_df = visits_df.fillna({"bill_amount": 0})
visits_filled_df.show()
visits_with_tax_df = visits_filled_df.withColumn("tax_amount", col("bill_amount") * 0.05)
visits_with_tax_df.show()
visits_final_bill_df = visits_with_tax_df.withColumn("final_bill", col("bill_amount") + col("tax_amount"))
visits_final_bill_df.show()

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|            2000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|            1000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|            2500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|            1200|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|            1800|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|            1100|
+---------+-----------+--------------+---------+----------------+----------------+

+--

doctor_id,doctor_name,specialization,city,experience_years,consultation_fee
D101,Dr. Ramesh,Cardiology,Hyderabad,12,1500
D106,Dr. Kiran,Cardiology,Hyderabad,20,3000


doctor_id,doctor_name,specialization,city,experience_years,consultation_fee
D101,Dr. Ramesh,Cardiology,Hyderabad,12,1500
D106,Dr. Kiran,Cardiology,Hyderabad,20,3000


doctor_id,doctor_name,specialization,city,experience_years,consultation_fee
D101,Dr. Ramesh,Cardiology,Hyderabad,12,1500
D104,Dr. Suresh,Orthopedics,Mumbai,15,2500
D106,Dr. Kiran,Cardiology,Hyderabad,20,3000


visit_id,patient_name,doctor_id,visit_date,diagnosis,bill_amount,payment_status
V1004,Sneha Patel,D104,2026-01-11,Fracture,12000,Paid
V1006,Neha Singh,D106,2026-01-12,Heart Checkup,7000,Paid
V1009,Kiran Rao,D104,2026-01-14,Back Pain,6000,Paid
V1010,Nisha Reddy,D101,2026-01-14,Heart Checkup,5500,Paid


...

lowest_bill_amount
1500


doctor_id,doctor_name,specialization,city,experience_years,consultation_fee
D106,Dr. Kiran,Cardiology,Hyderabad,20,3000
D104,Dr. Suresh,Orthopedics,Mumbai,15,2500
D102,Dr. Priya,Neurology,Bangalore,10,2000
D107,Dr. Farhan,Neurology,Pune,5,1800
D101,Dr. Ramesh,Cardiology,Hyderabad,12,1500
D105,Dr. Meera,Pediatrics,Delhi,6,1200
D108,Dr. Nisha,Dermatology,Kochi,9,1100
D103,Dr. Anita,Dermatology,Chennai,8,1000


visit_id,patient_name,doctor_id,visit_date,diagnosis,bill_amount,payment_status
V1004,Sneha Patel,D104,2026-01-11,Fracture,12000,Paid
V1006,Neha Singh,D106,2026-01-12,Heart Checkup,7000,Paid
V1009,Kiran Rao,D104,2026-01-14,Back Pain,6000,Paid
V1010,Nisha Reddy,D101,2026-01-14,Heart Checkup,5500,Paid
V1001,Rahul Sharma,D101,2026-01-10,Heart Checkup,5000,Paid
V1007,Arjun Verma,D120,2026-01-13,Migraine,4000,Paid
V1002,Priya Reddy,D102,2026-01-10,Migraine,3500,Paid
V1003,Amit Kumar,D103,2026-01-11,Skin Allergy,2000,Pending
V1005,Farhan Ali,D105,2026-01-12,Fever,1500,Paid
V1008,Meera Nair,D103,2026-01-13,Skin Allergy,null,Pending


visit_id,patient_name,doctor_id,visit_date,diagnosis,bill_amount,payment_status
V1008,Meera Nair,D103,2026-01-13,Skin Allergy,null,Pending


+--------+------------+---------+----------+-------------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+-------------+-----------+--------------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|          Paid|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|          Paid|
|   V1003|  Amit Kumar|     D103|2026-01-11| Skin Allergy|       2000|       Pending|
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|          Paid|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|       1500|          Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|          Paid|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|          Paid|
|   V1008|  Meera Nair|     D103|2026-01-13| Skin Allergy|          0|       Pending|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back P

Join Exercises


In [0]:
inner_join_df = doctors_df.join(visits_df, "doctor_id", "inner")
inner_join_df.show()
left_join_df = doctors_df.join(visits_df, "doctor_id", "left")
left_join_df.show()
right_join_df = doctors_df.join(visits_df, "doctor_id", "right")
right_join_df.show()
full_join_df = doctors_df.join(visits_df, "doctor_id", "full")
full_join_df.show()
invalid_doc_id = visits_filled_df.join(doctors_df, "doctor_id", "leftanti")
invalid_doc_id.show()

doc_without_visits = doctors_df.join(visits_df, "doctor_id", "leftanti")
doc_without_visits.show()
visits_per_doc = inner_join_df.groupBy("doctor_id", "doctor_name").count()
visits_per_doc.show()
doc_revenue = inner_join_df.groupBy("doctor_id", "doctor_name").agg(sum("bill_amount").alias("total_revenue"))
doc_revenue.show()
doc_revenue.orderBy(desc("total_revenue")).show(1)
spec_revenue = inner_join_df.groupBy("specialization").agg(sum("bill_amount").alias("total_revenue"))
spec_revenue.orderBy(desc("total_revenue")).show(1)

inner_join_df.groupBy("specialization").agg(avg("bill_amount").alias("avg_revenue")).show()
inner_join_df.groupBy("city").agg(sum("bill_amount").alias("total_revenue")).show()
visits_per_doc.show()
doc_revenue.orderBy(desc("total_revenue")).show(3)
doc_performance_report = inner_join_df.groupBy("doctor_id", "doctor_name","specialization").agg(count("visit_id").alias("total_visits"), sum("bill_amount").alias("total_revenue"), avg("bill_amount").alias("avg_revenue"))
doc_performance_report.show()

+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|visit_id|patient_name|visit_date|    diagnosis|bill_amount|payment_status|
+---------+-----------+--------------+---------+----------------+----------------+--------+------------+----------+-------------+-----------+--------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|   V1001|Rahul Sharma|2026-01-10|Heart Checkup|       5000|          Paid|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|            2000|   V1002| Priya Reddy|2026-01-10|     Migraine|       3500|          Paid|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|            1000|   V1003|  Amit Kumar|2026-01-11| Skin Allergy|       2000|       Pending|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|          

JSON Exercises


In [0]:
hospitals_df = spark.read.option("multiline", "true").json("/FileStore/tables/hospital_data/hospital_config.json")
hospitals_df.show()
hospitals_df.printSchema()
hos_phone_df = hospitals_df.withColumn("phone", col("contact.phone"))
hos_phone_df.show()
hos_email_df = hospitals_df.withColumn("email", col("contact.email"))
hos_email_df.show()
hospitals_df.filter(col("contact.phone").isNull()).show()

hospitals_df.filter(col("contact.email").isNull()).show()
hos_filled_df = hospitals_df.na.fill({"contact.phone": "Not Available"})
hos_filled_df = hospitals_df.na.fill({"contact.email": "Not Available"})
hos_filled_df.show()
hospitals_df.select("hospital_name", "city").show()
hospitals_df.select("hospital_name", "contact.phone").show()

explode_services_df = hospitals_df.withColumn("services", explode(col("services")))
explode_services_df.show()
explode_services_df.select("hospital_name", "services").show()
hospitals_df.groupBy("city").count().show()
explode_services_df.groupBy("services").count().show()
explode_services_df.filter(col("services") == "Cardiology").show()

explode_services_df.filter(col("services") == "Neurology").show()
explode_services_df.filter(col("services") == "Orthopedics").show()
explode_services_df.filter(col("services") == "Pediatrics").show()
flat_hospital_df = hospitals_df.select(col("hospital_id"), col("hospital_name"), col("city"), col("contact.phone").alias("phone"), col("contact.email").alias("email"), explode(col("services")).alias("service"))
flat_hospital_df.write.mode("overwrite").parquet("/FileStore/tables/hospital_data/transformed/hospitals.parquet")
flat_hospital_df.write.mode("overwrite").option("header", "true").csv("/FileStore/tables/hospital_data/transformed/hospitals.csv")

+---------+--------------------+-----------+----------------+--------------------+
|     city|             contact|hospital_id|   hospital_name|            services|
+---------+--------------------+-----------+----------------+--------------------+
|Hyderabad|{apollo@mail.com,...|       H101| Apollo Hospital|[Cardiology, Neur...|
|Hyderabad|{yashoda@mail.com...|       H102|Yashoda Hospital|[Cardiology, Orth...|
|Bangalore|  {NULL, 9876500013}|       H103|   Care Hospital|[Neurology, Pedia...|
+---------+--------------------+-----------+----------------+--------------------+

root
 |-- city: string (nullable = true)
 |-- contact: struct (nullable = true)
 |    |-- email: string (nullable = true)
 |    |-- phone: string (nullable = true)
 |-- hospital_id: string (nullable = true)
 |-- hospital_name: string (nullable = true)
 |-- services: array (nullable = true)
 |    |-- element: string (containsNull = true)

+---------+--------------------+-----------+----------------+-----------------

Window Function Exercises

In [0]:
from pyspark.sql.window import Window
revenue_df = inner_join_df.groupBy("doctor_id", "doctor_name", "specialization", "city").agg(sum("bill_amount").alias("revenue"))
w_desc = Window.orderBy(col("revenue").desc())
w_city_desc = Window.partitionBy("city").orderBy(col("revenue").desc())
w_spec_desc = Window.partitionBy("specialization").orderBy(col("revenue").desc())

revenue_df.withColumn("rank", rank().over(w_desc)).show()
revenue_df.withColumn("dense_rank", dense_rank().over(w_desc)).show()
revenue_df.withColumn("row_number", row_number().over(w_desc)).show()
revenue_df.withColumn("rank", rank().over(w_desc)).filter(col("rank")==1).show()
revenue_df.withColumn("rank", rank().over(w_desc)).filter(col("rank")<=3).show()

revenue_df.withColumn("rank", rank().over(w_spec_desc)).filter(col("rank")==1).show()
revenue_df.withColumn("rank", rank().over(w_spec_desc)).filter(col("rank")<=2).show()
window_running = Window.orderBy("doctor_id").rowsBetween(Window.unboundedPreceding, Window.currentRow)
revenue_df.withColumn("running_revenue", sum("revenue").over(window_running)).show()
w_lag_lead = Window.orderBy("doctor_id")
revenue_df.withColumn("prev_revenue", lag("revenue", 1).over(w_lag_lead)).show()
revenue_df.withColumn("next_revenue", lead("revenue", 1).over(w_lag_lead)).show()

revenue_df.withColumn("prev_revenue",lag("revenue",1).over(w_lag_lead)).withColumn("revenue_diff",col("revenue")-col("prev_revenue")).show()
revenue_df.withColumn("next_revenue",lead("revenue",1).over(w_lag_lead)).withColumn("revenue_diff",col("revenue")-col("next_revenue")).show()
revenue_df.withColumn("rank", rank().over(w_city_desc)).filter(col("rank")==1)
w_city = Window.partitionBy("city").orderBy("revenue")
revenue_df.withColumn("rank", rank().over(w_city)).filter(col("rank")==1).show()
revenue_df.withColumn("leaderboard", dense_rank().over(w_desc)).show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----+
|doctor_id|doctor_name|specialization|     city|revenue|rank|
+---------+-----------+--------------+---------+-------+----+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|   2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|   3|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|   4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|   5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|   6|
+---------+-----------+--------------+---------+-------+----+



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|dense_rank|
+---------+-----------+--------------+---------+-------+----------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|         1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|  10500|         2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|   7000|         3|
|     D102|  Dr. Priya|     Neurology|Bangalore|   3500|         4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|   2000|         5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|   1500|         6|
+---------+-----------+--------------+---------+-------+----------+

+---------+-----------+--------------+---------+-------+----------+
|doctor_id|doctor_name|specialization|     city|revenue|row_number|
+---------+-----------+--------------+---------+-------+----------+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|  18000|         1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad

Spark SQL Exercises

In [0]:
doctors_df.createOrReplaceTempView("doctors")
visits_df.createOrReplaceTempView("visits")
hospitals_df.createOrReplaceTempView("hospitals")
spark.sql("select * from doctors").show()
spark.sql("select specialization, count(*) from doctors group by specialization").show()

spark.sql("select city, count(*) from doctors group by city")
spark.sql("""SELECT d.doctor_id, d.doctor_name, SUM(v.bill_amount) as total_revenue 
        FROM doctors d JOIN visits v ON d.doctor_id = v.doctor_id GROUP BY d.doctor_id, d.doctor_name""").show()
spark.sql("""SELECT d.specialization, SUM(v.bill_amount) as total_revenue 
        FROM doctors d JOIN visits v ON d.doctor_id = v.doctor_id GROUP BY d.specialization""").show()
spark.sql("""SELECT d.doctor_id, d.doctor_name, SUM(v.bill_amount) as total_revenue 
    FROM doctors d JOIN visits v ON d.doctor_id = v.doctor_id 
    GROUP BY d.doctor_id, d.doctor_name ORDER BY total_revenue DESC LIMIT 5""").show()
spark.sql("SELECT * FROM visits WHERE payment_status='Pending'").show()

spark.sql("SELECT * FROM hospitals WHERE array_contains(services,'Cardiology')").show()
spark.sql("SELECT * FROM hospitals WHERE array_contains(services,'Neurology')").show()
spark.sql("SELECT * FROM hospitals WHERE contact.phone IS NULL OR contact.email IS NULL").show()
spark.sql("SELECT avg(consultation_fee) AS avg_consultation_fee FROM doctors").show()
spark.sql("""select d.doctor_id, d.doctor_name, COUNT(v.visit_id) as total_visits, SUM(v.bill_amount) as total_revenue
    FROM doctors d JOIN visits v ON d.doctor_id = v.doctor_id
    GROUP BY d.doctor_id, d.doctor_name""").show()

+---------+-----------+--------------+---------+----------------+----------------+
|doctor_id|doctor_name|specialization|     city|experience_years|consultation_fee|
+---------+-----------+--------------+---------+----------------+----------------+
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|              12|            1500|
|     D102|  Dr. Priya|     Neurology|Bangalore|              10|            2000|
|     D103|  Dr. Anita|   Dermatology|  Chennai|               8|            1000|
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|              15|            2500|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|               6|            1200|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|              20|            3000|
|     D107| Dr. Farhan|     Neurology|     Pune|               5|            1800|
|     D108|  Dr. Nisha|   Dermatology|    Kochi|               9|            1100|
+---------+-----------+--------------+---------+----------------+----------------+

+--

ETL Exercises


In [0]:
r_doctors = spark.read.option("header", "true").option("inferSchema", "true").csv("/FileStore/tables/hospital_data/doctors.csv")
r_visits = spark.read.option("header", "true").option("inferSchema", "true").csv("/FileStore/tables/hospital_data/visits.csv")
r_hospitals = spark.read.option("multiline", "true").json("/FileStore/tables/hospital_data/hospital_config.json")

cln_visits = r_visits.na.fill({"bill_amount":0})
cln_visits.show()

flat_hospitals = r_hospitals.select(
    col("hospital_id"), col("hospital_name"), col("city"),col("contact.phone").alias("phone"), col("contact.email").alias("email"),
    explode(col("services")).alias("service"))
flat_hospitals.show()

dv_df = r_doctors.join(cln_visits, "doctor_id", "inner")
dv_df.show()

rev_calc = dv_df.groupBy("doctor_id", "doctor_name", "specialization", "city").agg(sum("bill_amount").alias("total_revenue"), count("visit_id").alias("total_visits"))
rev_calc.show()

window = Window.orderBy(desc("total_revenue"))
rev_calc = rev_calc.withColumn("rank", dense_rank().over(window))
rev_calc.show()

spec_summary = rev_calc.groupBy("specialization").agg(sum("total_revenue").alias("total_revenue"), avg("total_revenue").alias("avg_revenue"))
spec_summary.show()

rev_calc.write.mode("overwrite").parquet("/FileStore/tables/hospital_data/silver/doctor_revenue")

gold_report = rev_calc.filter(col("rank")<=3)
gold_report.write.mode("overwrite").parquet("/FileStore/tables/hospital_data/gold/doctor_performance")

dashboard = rev_calc.join(flat_hospitals, (rev_calc.city==flat_hospitals.city)&(rev_calc.specialization == flat_hospitals.service), "full")
dashboard.show()

+--------+------------+---------+----------+-------------+-----------+--------------+
|visit_id|patient_name|doctor_id|visit_date|    diagnosis|bill_amount|payment_status|
+--------+------------+---------+----------+-------------+-----------+--------------+
|   V1001|Rahul Sharma|     D101|2026-01-10|Heart Checkup|       5000|          Paid|
|   V1002| Priya Reddy|     D102|2026-01-10|     Migraine|       3500|          Paid|
|   V1003|  Amit Kumar|     D103|2026-01-11| Skin Allergy|       2000|       Pending|
|   V1004| Sneha Patel|     D104|2026-01-11|     Fracture|      12000|          Paid|
|   V1005|  Farhan Ali|     D105|2026-01-12|        Fever|       1500|          Paid|
|   V1006|  Neha Singh|     D106|2026-01-12|Heart Checkup|       7000|          Paid|
|   V1007| Arjun Verma|     D120|2026-01-13|     Migraine|       4000|          Paid|
|   V1008|  Meera Nair|     D103|2026-01-13| Skin Allergy|          0|       Pending|
|   V1009|   Kiran Rao|     D104|2026-01-14|    Back P

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+---------+-----------+--------------+---------+-------------+------------+----+
|doctor_id|doctor_name|specialization|     city|total_revenue|total_visits|rank|
+---------+-----------+--------------+---------+-------------+------------+----+
|     D104| Dr. Suresh|   Orthopedics|   Mumbai|        18000|           2|   1|
|     D101| Dr. Ramesh|    Cardiology|Hyderabad|        10500|           2|   2|
|     D106|  Dr. Kiran|    Cardiology|Hyderabad|         7000|           1|   3|
|     D102|  Dr. Priya|     Neurology|Bangalore|         3500|           1|   4|
|     D103|  Dr. Anita|   Dermatology|  Chennai|         2000|           2|   5|
|     D105|  Dr. Meera|    Pediatrics|    Delhi|         1500|           1|   6|
+---------+-----------+--------------+---------+-------------+------------+----+

+--------------+-------------+-----------+
|specialization|total_revenue|avg_revenue|
+--------------+-------------+-----------+
|    Cardiology|        17500|     8750.0|
|   Orthopedics| 